# Simulation setup

In [1]:
import os
import json

with open(".env.json", "r") as f_in:
    env_json: dict = json.load(f_in)
    for key, value in env_json.items():
        os.environ[key] = value

# Example: Hello world

In [28]:
import boto3
import json

aws_access_key = os.environ['aws_access_key']
aws_secret_key = os.environ['aws_secret_key']
aws_region = os.environ['aws_region']

def lambda_handler(event, context):
    
    event_method = event["httpMethod"]
    
    if event_method != 'POST':
        return {
            'statusCode': 200,
            'body': json.dumps("Not implemented")
        }
        
    event_body = event["body"]

    prompt_data = event_body["prompt"]
    
    client_bedrock_runtime = boto3.client(
        service_name="bedrock-runtime",
        region_name=aws_region,
        aws_access_key_id=aws_access_key,
        aws_secret_access_key=aws_secret_key
    )
    
    
    # Depending on model payload may look different
    payload = {
        "inputText": f"\n\nUser:{prompt_data}\n\nAsistante:", 
        "textGenerationConfig": {
            "maxTokenCount": 256,
            "stopSequences": ["User:"],
            "temperature": 0.4,
            "topP": 0.9
        }
    }
    
    body = json.dumps(payload)

    model_id = 'amazon.titan-text-express-v1'

    response = client_bedrock_runtime.invoke_model(
        body=body,
        modelId=model_id,
        accept="application/json",
        contentType="application/json",
    )
    
    response_body = json.loads(response.get("body").read())
    response_text = response_body.get("results")[0]["outputText"]  # response_body.get("completions")
    # print(response_text)

    # print(response)
    print(response_body)
    
    return {
        'statusCode': 200,
        'body': json.dumps(response_text)
    }

In [30]:
event = {
    "httpMethod": "POST",
    "body": {
        "prompt": "hola como estas"
    }
}
lambda_response = lambda_handler(event, context=None)
print(lambda_response)
# print(json.loads(lambda_response['body'])[0]["data"]["text"])

{'inputTextTokenCount': 16, 'results': [{'tokenCount': 18, 'outputText': 'Hola, ¿en qué le puedo ayudar?\n\n', 'completionReason': 'STOP_CRITERIA_MET'}]}
{'statusCode': 200, 'body': '"Hola, \\u00bfen qu\\u00e9 le puedo ayudar?\\n\\n"'}


In [32]:
print(json.loads(lambda_response["body"]))

Hola, ¿en qué le puedo ayudar?




# END